# Frueherkennung von Pumpenausfaellen aus Sensordaten
**Autorin:** Mersiha Alic

**Frage:** Kann man aus den Sensordaten einer Industriepumpe einen Ausfall
erkennen, *bevor* er passiert - und wie viele Stunden Vorwarnzeit gewinnt man?

**Daten:** Kaggle "Pump Sensor Data" (nphantawee/pump-sensor-data), Datei `sensor.csv`:
52 Sensoren, Messung jede Minute, ca. 5 Monate, 7 Ausfaelle.
Spalte `machine_status`: NORMAL, BROKEN (Ausfall) oder RECOVERING (Wiederanlauf).

**Vorgehen**
1. Daten pruefen und bereinigen
2. Sensorverlauf und Betriebszustaende darstellen
3. Health Score (Stundenwerte): Vorwarnzeit je Ausfall
4. Kurzfristige Vorwarnung (Minutenwerte, letzte 6 Stunden)
5. Machine-Learning-Modell: "Ausfall in den naechsten 24 Stunden?" (Random Forest)
6. Fazit


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

import os
DATEI = "/content/sensor.csv" 
VORHERSAGE_H = 24        # "Ausfall in den naechsten X Stunden"
FENSTER_H = 48           # Zeitraum vor einem Ausfall, in dem eine Warnung als "rechtzeitig" zaehlt
TRAININGSTAGE = 10       # erste Tage im Normalbetrieb = Referenz fuer den Health Score

## 1. Daten laden und pruefen

In [ ]:
df = pd.read_csv(DATEI, parse_dates=["timestamp"])
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
df = df.set_index("timestamp").sort_index()

print(f"Zeitraum: {df.index.min():%d.%m.%Y} bis {df.index.max():%d.%m.%Y}, {len(df):,} Messungen")
print("\nBetriebszustaende:")
print(df["machine_status"].value_counts())

sensoren = [c for c in df.columns if c.startswith("sensor_")]
fehlend = df[sensoren].isna().mean().sort_values(ascending=False)
print("\nSensoren mit den meisten fehlenden Werten (Anteil):")
print(fehlend.head(5).round(3))

## Bereinigung
- Sensoren mit mehr als 30 % fehlenden Werten werden entfernt (im Kaggle-Datensatz
  sind das `sensor_15` (komplett leer) und `sensor_50`).
- Einzelne Luecken werden mit dem letzten gueltigen Wert gefuellt (Sensor haelt Wert).

In [ ]:
entfernen = fehlend[fehlend > 0.30].index.tolist()
print("Entfernt:", entfernen)
sensoren = [s for s in sensoren if s not in entfernen]
df = df.drop(columns=entfernen)
df[sensoren] = df[sensoren].ffill().bfill()

ausfaelle = df.index[df["machine_status"] == "BROKEN"]
print(f"\n{len(ausfaelle)} Ausfaelle:")
for t in ausfaelle:
    print("  ", f"{t:%d.%m.%Y %H:%M}")

## 2. Sensorverlauf und Betriebszustaende
Die vier Sensoren, die sich in den 24 h vor einem Ausfall am staerksten veraendern.
Rote Linie = Ausfall, grau = Wiederanlauf.

**Beobachtung:** Einige Sensoren (z. B. `sensor_36`) springen zwischen festen Niveaus - die Pumpe
laeuft offenbar in mehreren **Betriebszustaenden**. Ein Wechsel des Betriebszustands sieht fuer
einen einfachen Health Score wie eine Abweichung aus und erhoeht das Grundrauschen.

In [ ]:
def markiere_ausfaelle(ax):
    for t in ausfaelle:
        ax.axvline(t, color="#c0392b", lw=1.2)
    rec = (df["machine_status"] == "RECOVERING").astype(int)
    ax.fill_between(df.index, 0, 1, where=rec == 1, color="grey", alpha=0.15,
                    transform=ax.get_xaxis_transform())

# Sensoren, die sich in den 24 h VOR einem Ausfall am staerksten vom uebrigen Normalbetrieb unterscheiden
vor24 = pd.Series(False, index=df.index)
for t in ausfaelle:
    vor24.loc[t - pd.Timedelta(hours=24): t] = True
ist_normal = df["machine_status"] == "NORMAL"
normal = df.loc[ist_normal & ~vor24, sensoren]
vorher = df.loc[ist_normal & vor24, sensoren]
trenn = ((vorher.mean() - normal.mean()).abs() / normal.std()).sort_values(ascending=False)
zeige = trenn.index[:4].tolist()

taeglich = df[zeige].resample("1h").mean()
fig, axes = plt.subplots(len(zeige), 1, figsize=(12, 8), sharex=True)
for ax, s in zip(axes, zeige):
    ax.plot(taeglich.index, taeglich[s], lw=0.8, color="#1f6f8b")
    markiere_ausfaelle(ax)
    ax.set_ylabel(s, rotation=0, ha="right")
axes[0].set_title("Sensorwerte (Stundenmittel) - rote Linie = Ausfall, grau = Wiederanlauf")
plt.tight_layout()
plt.savefig("grafik1_sensoren_ausfaelle.png", dpi=150)
plt.show()

## 3. Health Score und Vorwarnzeit
Die Pumpe wird mit ihrem **eigenen Normalbetrieb der letzten 7 Tage** verglichen
(gleitender Median, ohne die letzten 2 Tage und ohne Wiederanlaufphasen). So stoert es nicht,
wenn sich Werte ueber Monate langsam veraendern - auffaellig ist nur eine *neue* Abweichung.
Fuer jeden Sensor: Abstand zum Referenzwert, geteilt durch seine normale Streuung (IQR).
Der **Health Score** ist der Mittelwert ueber alle Sensoren (Stundenwerte).
**Warnschwelle**: so gewaehlt, dass die Pumpe im Normalbetrieb hoechstens 1 % der Zeit warnt.

In [ ]:
stunde = df[sensoren].resample("1h").mean()
status_h = df["machine_status"].resample("1h").agg(
    lambda s: "BROKEN" if (s == "BROKEN").any() else ("RECOVERING" if (s == "RECOVERING").any() else "NORMAL"))
nur_normal = stunde.where(status_h == "NORMAL")
referenz = nur_normal.rolling("7D", min_periods=48).median().shift(48)   # 48 h Abstand
start = df.index.min()
erste_tage = nur_normal[nur_normal.index < start + pd.Timedelta(days=TRAININGSTAGE)]
iqr = (erste_tage.quantile(0.75) - erste_tage.quantile(0.25)).replace(0, np.nan)
z = ((stunde - referenz) / iqr).abs().clip(upper=20)
health = z.mean(axis=1)
schwelle = health[status_h == "NORMAL"].quantile(0.99)
warnung = (health > schwelle) & (status_h == "NORMAL")
print(f"Warnschwelle Health Score: {schwelle:.2f}")

ergebnisse = []
for t in ausfaelle:
    fenster = warnung.loc[t - pd.Timedelta(hours=FENSTER_H): t]
    if fenster.any():
        erste = fenster.idxmax()
        vorlauf = (t - erste).total_seconds() / 3600
    else:
        vorlauf = 0.0
    ergebnisse.append({"Ausfall": t, "Vorwarnzeit_h": round(vorlauf, 1)})
vorwarn = pd.DataFrame(ergebnisse)
print("\nVorwarnzeit je Ausfall (Stunden, max. 48):")
print(vorwarn.to_string(index=False))

# Fehlalarme: Warnungen im Normalbetrieb, die NICHT in den 48 h vor einem Ausfall liegen
vor_ausfall = pd.Series(False, index=health.index)
for t in ausfaelle:
    vor_ausfall.loc[t - pd.Timedelta(hours=FENSTER_H): t] = True
normalbetrieb = (status_h == "NORMAL") & ~vor_ausfall
fehlalarm_h = (warnung & normalbetrieb).sum()          # Stundenwerte
wochen = normalbetrieb.sum() / 24 / 7
print(f"\nFehlalarm-Zeit: {fehlalarm_h:.0f} h in {wochen:.0f} Wochen Normalbetrieb "
      f"({fehlalarm_h / wochen:.1f} h pro Woche)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
h = health.where(status_h == "NORMAL")
ax.plot(h.index, h, lw=0.8, color="#1f6f8b", label="Health Score (Stundenwerte, Normalbetrieb)")
ax.axhline(schwelle, color="#e67e22", ls="--", lw=1.2, label="Warnschwelle")
markiere_ausfaelle(ax)
ax.set_ylim(0, max(schwelle * 2.5, h.quantile(0.995)))
ax.set_ylabel("Abweichung vom Normalbetrieb")
erkannt = (vorwarn["Vorwarnzeit_h"] > 0).sum()
mittel = vorwarn.loc[vorwarn["Vorwarnzeit_h"] > 0, "Vorwarnzeit_h"].median() if erkannt else 0
ax.set_title(f"Health Score: {erkannt} von {len(vorwarn)} Ausfaellen vorher erkannt, "
             f"Vorwarnzeit (Median der erkannten) {mittel:.0f} h")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
plt.tight_layout()
plt.savefig("grafik2_health_score.png", dpi=150)
plt.show()

## 4. Kurzfristige Vorwarnung (Minutenwerte)
Kuendigt sich ein Ausfall wenigstens **Minuten bis wenige Stunden** vorher an?
Health Score auf Minutenbasis (geglaettet ueber 10 Minuten), Referenz = gesamter Normalbetrieb.
Fast konstante Sensoren und Sensoren, die schon im Normalbetrieb oft "ausschlagen", werden
ausgeschlossen. Warnschwelle = 99,9 %-Perzentil des Normalbetriebs (fern von Ausfaellen).

In [ ]:
nn = df.loc[df["machine_status"] == "NORMAL", sensoren]
med = nn.median()
mad_m = (nn - med).abs().median() * 1.4826
brauchbar = mad_m[mad_m > 1e-6].index
zm = ((df[brauchbar] - med[brauchbar]) / mad_m[brauchbar]).abs().clip(upper=20)
zm = zm.loc[:, (zm.loc[nn.index] >= 20).mean() < 0.05]
hs = zm.mean(axis=1).rolling("10min").mean()
normal_ok = df["machine_status"] == "NORMAL"
fern = pd.Series(True, index=df.index)
for t in ausfaelle:
    fern.loc[t - pd.Timedelta(hours=6): t] = False
schwelle_m = hs[normal_ok & fern].quantile(0.999)
print(f"{zm.shape[1]} Sensoren genutzt, Warnschwelle: {schwelle_m:.2f}")

zeilen = []
for t in ausfaelle:
    f = hs.loc[t - pd.Timedelta(hours=6): t - pd.Timedelta(minutes=1)]
    ueber = f > schwelle_m
    zeilen.append({"Ausfall": f"{t:%d.%m.%Y %H:%M}",
                   "Vorwarnung_min": int((t - ueber.idxmax()).total_seconds() / 60) if ueber.any() else 0})
kurz = pd.DataFrame(zeilen)
fehlalarm_kurz = (hs > schwelle_m)[normal_ok & fern].mean() * 7 * 24
print(kurz.to_string(index=False))
print(f"Fehlalarm-Zeit: {fehlalarm_kurz:.1f} h pro Woche")

In [ ]:
fig, axes = plt.subplots(len(ausfaelle), 1, figsize=(10, 1.6 * len(ausfaelle)), sharex=True)
for ax, t, v in zip(axes, ausfaelle, kurz["Vorwarnung_min"]):
    s = hs.loc[t - pd.Timedelta(hours=6): t + pd.Timedelta(minutes=30)]
    ax.plot((s.index - t).total_seconds() / 3600, s.values, color="#1f6f8b", lw=1)
    ax.axhline(schwelle_m, color="#e67e22", ls="--", lw=1)
    ax.axvline(0, color="#c0392b", lw=1.2)
    ax.set_ylabel(f"{t:%d.%m.}", rotation=0, ha="right")
    ax.text(0.01, 0.8, f"Vorwarnung: {v} min" if v else "keine Vorwarnung",
            transform=ax.transAxes, fontsize=8, color="#444")
axes[-1].set_xlabel("Stunden relativ zum Ausfall (0 = Ausfall)")
axes[0].set_title("Health Score in den 6 Stunden vor jedem Ausfall (orange = Warnschwelle)")
plt.tight_layout()
plt.savefig("grafik3_letzte_6_stunden.png", dpi=150)
plt.show()

## 5. Modell: Ausfall in den naechsten 24 Stunden?
- Aufloesung 10 Minuten; Merkmale = **Abweichung** jedes Sensors vom Normalbetrieb der letzten
  3 Tage (statt absoluter Werte, die sich ueber Monate verschieben) plus Streuung ueber 6 h
- Zielvariable = 1, wenn innerhalb der naechsten 24 h ein Ausfall folgt; Wiederanlauf ausgeschlossen
- **Leave-one-failure-out**: Zeitachse in 7 Bloecke mit je einem Ausfall. Das Modell wird
  7-mal auf 6 Bloecken trainiert und auf dem jeweils fehlenden Block getestet.

In [ ]:
d10 = df[sensoren].resample("10min").mean()
status10 = df["machine_status"].resample("10min").agg(
    lambda s: "BROKEN" if (s == "BROKEN").any() else ("RECOVERING" if (s == "RECOVERING").any() else "NORMAL"))
ziel = pd.Series(0, index=d10.index)
for t in ausfaelle:
    ziel.loc[t - pd.Timedelta(hours=VORHERSAGE_H): t] = 1

ist_n = status10 == "NORMAL"
d10n = d10.where(ist_n)
basis = d10n.rolling("3D", min_periods=144).median().shift(144)
mad10 = (d10n - d10n.median()).abs().median().replace(0, np.nan) * 1.4826
abw = ((d10 - basis) / mad10).clip(-20, 20)
Xr = pd.concat([abw.add_suffix("_abw"),
                abw.rolling(36).mean().add_suffix("_abw6h"),
                (d10.rolling(36).std() / mad10).clip(upper=20).add_suffix("_std6h")], axis=1)
ok = ist_n & Xr.notna().all(axis=1)
Xr, yr = Xr[ok].fillna(0), ziel[ok]

grenzen = [Xr.index.min()] + [ausfaelle[i] + (ausfaelle[i + 1] - ausfaelle[i]) / 2
                             for i in range(len(ausfaelle) - 1)] + [Xr.index.max() + pd.Timedelta(minutes=10)]
zeilen, alle_p, alle_y = [], [], []
for k, t in enumerate(ausfaelle):
    test = (Xr.index >= grenzen[k]) & (Xr.index < grenzen[k + 1])
    m = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=20,
                               class_weight="balanced", n_jobs=-1, random_state=42)
    m.fit(Xr[~test], yr[~test])
    p = pd.Series(m.predict_proba(Xr[test])[:, 1], index=Xr.index[test])
    alle_p.append(p); alle_y.append(yr[test])
    alarm = p > 0.5
    vor = alarm.loc[t - pd.Timedelta(hours=VORHERSAGE_H): t]
    rest = alarm[yr[test] == 0]
    zeilen.append({"Ausfall": f"{t:%d.%m.%Y}", "erkannt": bool(vor.any()),
                   "Vorwarnzeit_h": round((t - vor.idxmax()).total_seconds() / 3600, 1) if vor.any() else 0.0,
                   "Fehlalarm_h_pro_Woche": round(rest.sum() / 6 / (len(rest) / 6 / 24 / 7), 1)})
lofo = pd.DataFrame(zeilen)
print(lofo.to_string(index=False))
P, Y = pd.concat(alle_p), pd.concat(alle_y)
print(classification_report(Y, P > 0.5, target_names=["kein Ausfall", "Ausfall in 24 h"], zero_division=0))

## 6. Fazit
Die folgende Zelle fasst die Ergebnisse automatisch zusammen.

In [ ]:
erk_h = vorwarn[vorwarn["Vorwarnzeit_h"] > 0]
erk_k = kurz[kurz["Vorwarnung_min"] > 0]
print("ERGEBNISSE")
print(f"- Health Score (Stunden, 48-h-Fenster): {len(erk_h)} von {len(vorwarn)} Ausfaellen vorher erkannt"
      + (f", Vorwarnzeit {', '.join(f'{v:.0f} h' for v in erk_h['Vorwarnzeit_h'])}" if len(erk_h) else "")
      + f"; Fehlalarm-Zeit {fehlalarm_h / wochen:.1f} h pro Woche")
print(f"- Kurzfristig (Minuten, 6-h-Fenster): {len(erk_k)} von {len(kurz)} Ausfaellen vorher erkannt"
      + (f", Median {erk_k['Vorwarnung_min'].median():.0f} min" if len(erk_k) else "")
      + f"; Fehlalarm-Zeit {fehlalarm_kurz:.1f} h pro Woche")
print(f"- Modell 'Ausfall in 24 h' (leave-one-failure-out): {lofo['erkannt'].sum()} von {len(lofo)} Ausfaellen erkannt")

**Interpretation**
- In diesen Daten kuendigen sich die meisten Ausfaelle **nicht** Stunden vorher an - sie kommen
  abrupt. Nur einzelne Ausfaelle (v. a. Ende Juni) zeigen eine klare, laengere Vorwarnung.
- Die Pumpe laeuft in mehreren Betriebszustaenden. Ein naechster Schritt waere ein Health Score
  **je Betriebszustand** (z. B. Zustaende per Clustering erkennen).
- Mit nur 7 Ausfaellen ist das ein Machbarkeitsnachweis, keine belastbare Statistik.
  Viele Kaggle-Loesungen mit ~99 % Accuracy erkennen nur, dass die Pumpe *bereits* ausgefallen ist.

